[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1vYC7dQwbcR0Fz9hfiwnkYBOWN22_qOlb#scrollTo=U8TZYWK0OeEQ)

# Building a RAG Pipeline from Scratch (No Frameworks)

> A complete, hands-on tutorial for building a **Retrieval-Augmented Generation (RAG)**
> pipeline using only raw Python libraries — no LangChain, no LlamaIndex, no abstractions.
> Every step is explained with *what*, *why*, and *what else you could do instead*.

---

## What is RAG and Why Does It Exist?

Large Language Models (LLMs) are trained on a fixed snapshot of the internet up to some
cutoff date. They cannot know about your private documents, recent regulations, or
internal company policies. They also **hallucinate** — they confidently generate plausible
but wrong answers when they don't know something.

**RAG solves both problems** by:
1. Searching your own documents for relevant context at query time
2. Injecting that context into the LLM prompt so it answers *from your data*, not from memory

The result: an LLM that behaves like an expert who has *just read* the relevant section
of your document before answering.


```
User Query
    │
    ▼
[Embed query] ──► [Search vector store] ──► [Top-k chunks retrieved]
                                                       │
                                                       ▼
                                          [Inject into LLM prompt]
                                                       │
                                                       ▼
                                              [Grounded answer]
```

---

## Libraries We Use (and Why These, Not Others)

| Library | Role | Why not the alternative? |
|---|---|---|
| `pypdf` | Extract text from PDFs | `pdfminer` is more accurate but heavier; `pymupdf` is fastest but needs C libs |
| `sentence-transformers` | Convert text → vectors | OpenAI embeddings cost money; this runs free, locally |
| `numpy` | Vector math (cosine similarity) | Core scientific computing, no abstraction overhead |
| `faiss-cpu` | Fast similarity search index | `annoy` is simpler; FAISS is production-grade and from Meta |
| `requests` | Download PDFs | `urllib` gets blocked by servers; `requests` mimics browsers |

Install everything at once:

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu numpy requests openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.8 MB/s eta 0:00:00



---

## Part 1 — Document Acquisition

### Documents We Will Use

These are all **free, public, no-login-required** regulatory and legal documents.
They are ideal for RAG because they are long, dense, structured, and authoritative.

In [2]:
import requests
import os

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

DOCUMENTS = {
    # NIST AI Risk Management Framework 1.0 (2023)
    # ~50 pages | Covers AI governance, risk mapping, measurement, management
    "nist_ai_rmf.pdf": "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf",

    # NIST Generative AI Profile (AI 600-1) (2024)
    # Companion to AI RMF, focused on GenAI-specific risks
    "nist_genai_profile.pdf": "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf",

    # EU AI Act — Official Journal version (July 2024)
    # World's first comprehensive AI law, ~144 pages, rich regulatory language
    "eu_ai_act.pdf": "https://artificialintelligenceact.eu/wp-content/uploads/2024/04/TA-9-2024-0138_EN.pdf",

    # CFPB Supervision & Examination Manual
    # US financial consumer protection guidance, very long-form (~900 pages)
    # Great for stress-testing your chunking and retrieval at scale
    "cfpb_manual.pdf": "https://files.consumerfinance.gov/f/documents/cfpb_supervision-and-examination-manual.pdf",
}

os.makedirs("docs", exist_ok=True)

for filename, url in DOCUMENTS.items():
    path = f"docs/{filename}"
    if os.path.exists(path):
        print(f"✓ Already exists: {filename}")
        continue
    print(f"Downloading {filename}...")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=90, stream=True)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        size_kb = os.path.getsize(path) // 1024
        print(f"  ✓ {size_kb} KB saved")
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        # Fallback: use wget which bypasses Python user-agent issues
        print(f"  → Try: !wget -q --user-agent='Mozilla/5.0' '{url}' -O {path}")

  ✓ 1900 KB saved
  ✓ 1147 KB saved
  ✓ 842 KB saved
  ✓ 11915 KB saved


**Why these documents specifically?**
- They are long enough (50–900 pages) to make retrieval meaningful
- They have varied structure: tables, numbered articles, definitions, recitals
- They represent a real use case: legal/compliance Q&A bots are one of the most
  common enterprise RAG applications

---

## Part 2 — PDF Parsing

### What We Are Doing

Extract raw text from each PDF page-by-page. We preserve page numbers in metadata
because when we return answers to users, we want to cite the exact source page.

In [3]:
import pypdf
import os

def load_pdf(filepath: str) -> list[dict]:
    """
    Load a PDF and return a list of page dicts.
    Each dict has: text, page_number, source_file
    """
    pages = []
    reader = pypdf.PdfReader(filepath)
    filename = os.path.basename(filepath)

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()

        # Skip empty pages (common in scanned PDFs with images only)
        if not text or len(text.strip()) < 50:
            continue

        pages.append({
            "text": text.strip(),
            "page_number": page_num + 1,   # 1-indexed, human-friendly
            "source_file": filename,
        })

    return pages


# Load all downloaded PDFs
all_pages = []
for filename in os.listdir("docs"):
    if not filename.endswith(".pdf"):
        continue
    filepath = f"docs/{filename}"
    pages = load_pdf(filepath)
    all_pages.extend(pages)
    print(f"{filename}: {len(pages)} pages extracted")

print(f"\nTotal pages across all documents: {len(all_pages)}")

cfpb_manual.pdf: 1814 pages extracted
nist_genai_profile.pdf: 64 pages extracted
nist_ai_rmf.pdf: 48 pages extracted
eu_ai_act.pdf: 459 pages extracted

Total pages across all documents: 2385


### What Could Go Wrong Here

| Problem | Cause | Fix |
|---|---|---|
| Empty text on pages | PDF is image-based (scanned) | Use `pytesseract` OCR instead |
| Garbled text | Unusual PDF font encoding | Switch to `pymupdf` (`fitz`) |
| Huge memory usage | Very large PDFs | Process one file at a time, stream pages |

**Alternative parsers:**
- `pymupdf` (fitz) — faster, handles more PDF types, but requires compiled C library
- `pdfplumber` — better at extracting tables from PDFs
- `pytesseract` — for scanned/image PDFs using OCR

---

## Part 3 — Chunking

### Why Chunking Matters

You cannot embed an entire 100-page document as one vector — the semantic meaning gets
averaged and diluted into mush. You also cannot feed 100 pages into an LLM prompt —
context windows have limits and cost tokens.

**Chunking** breaks documents into small, semantically coherent pieces that can be:
- Individually embedded into a precise vector
- Retrieved selectively (only the relevant 3–5 chunks, not the whole document)
- Fit inside an LLM prompt alongside a user question

### The Sliding Window Approach

We use a **character-level sliding window** with overlap. The overlap is critical —
it ensures that a sentence split across a chunk boundary is still fully represented
in at least one chunk.

```
Page text: "...Article 9 requires risk assessment. The assessment must cover all phases..."
                                              │
               chunk 1 ends ────────────────►│◄──── chunk 2 begins (50-char overlap)
```

In [4]:
def chunk_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 150,
) -> list[str]:
    """
    Split text into overlapping chunks by character count.

    chunk_size: target characters per chunk (~200-250 words)
    overlap: characters shared between consecutive chunks
             prevents losing context at boundaries
    """
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        # If not at end of text, try to break at a sentence boundary
        # (avoid cutting mid-sentence)
        if end < len(text):
            # Look backwards for the last period within the chunk
            last_period = text.rfind(".", start, end)
            if last_period != -1 and last_period > start + chunk_size // 2:
                end = last_period + 1  # include the period

        chunk = text[start:end].strip()
        if len(chunk) > 50:   # discard tiny fragments
            chunks.append(chunk)

        start = end - overlap  # slide back by overlap amount

    return chunks


def chunk_pages(
    pages: list[dict],
    chunk_size: int = 1000,
    overlap: int = 150,
) -> list[dict]:
    """
    Chunk all pages and carry forward source metadata.
    Returns list of chunk dicts with text + provenance info.
    """
    all_chunks = []
    for page in pages:
        chunks = chunk_text(page["text"], chunk_size, overlap)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "source_file": page["source_file"],
                "page_number": page["page_number"],
                "chunk_index": i,
            })
    return all_chunks


chunks = chunk_pages(all_pages, chunk_size=1000, overlap=150)

print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(c['text']) for c in chunks) // len(chunks)} chars")
print(f"\nSample chunk from '{chunks[20]['source_file']}' page {chunks[20]['page_number']}:")
print("-" * 60)
print(chunks[20]["text"][:500])

Total chunks: 7731
Average chunk length: 772 chars

Sample chunk from 'cfpb_manual.pdf' page 7:
------------------------------------------------------------
ntial violations of Federal consumer financial laws. In this Manual the coordination 
process will generally be referred to as “consulting internally.” Alternatively, “Headquarters” will 
be used to signify the involvement of multiple divisions or offices in addition to Supervision. 
Specific examination procedures will be similar to those of the prudential and, in some instances, 
State regulators.
13As appropriate and in accordance with CFPB policy, examiners and 
Supervision managers will gen


### Chunking Strategies Compared

| Strategy | How it works | Best for |
|---|---|---|
| **Fixed character window** ← *we use this* | Slice at N chars, slide by overlap | Simple, fast, works everywhere |
| **Sentence-aware** | Split on `.`, `?`, `!` using nltk | Better coherence, slightly slower |
| **Paragraph-aware** | Split on `\n\n` | Well-structured documents (legal, academic) |
| **Semantic chunking** | Embed sentences, split where embedding similarity drops | Best quality, expensive to compute |
| **Recursive splitter** | Try paragraph → sentence → word until fits | LangChain's default, very robust |

**Rule of thumb:** chunk_size=500–1500 characters, overlap=10–15% of chunk size.
For legal text (dense, long sentences), lean toward larger chunks (1000–1500).

---

## Part 4 — Embeddings

### What Is an Embedding?

An embedding is a list of numbers (a vector) that represents the **meaning** of a piece
of text in high-dimensional space. Texts with similar meaning end up close together in
this space. This is what allows us to search by meaning, not just keywords.

```
"What are the AI Act obligations?" ──► [0.23, -0.81, 0.44, ...]  (384 numbers)
"Providers must comply with Article 9" ──► [0.21, -0.79, 0.47, ...]  (close!)
"The weather is nice today" ──► [-0.55, 0.33, -0.12, ...]  (far away)
```

### Choosing an Embedding Model

We use `all-MiniLM-L6-v2` — a free, fast, local model from HuggingFace that:
- Runs entirely on CPU (no GPU needed)
- Produces 384-dimensional vectors
- Downloads once (~90MB) and is cached

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model (downloads on first run, ~90MB)...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_chunks(chunks: list[dict], batch_size: int = 64) -> np.ndarray:
    """
    Embed all chunk texts and return a 2D numpy array.
    Shape: (num_chunks, embedding_dim) → e.g. (3200, 384)

    We batch to avoid memory spikes on large document sets.
    """
    texts = [c["text"] for c in chunks]
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        embeddings = model.encode(
            batch,
            convert_to_numpy=True,
            show_progress_bar=False,
            normalize_embeddings=True,  # L2 normalize → cosine sim = dot product
        )
        all_embeddings.append(embeddings)
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks...", end="\r")

    print()
    return np.vstack(all_embeddings)


print("Embedding all chunks...")
embeddings = embed_chunks(chunks)
print(f"Embedding matrix shape: {embeddings.shape}")
# e.g. (3200, 384) → 3200 chunks, each a 384-dim vector

Loading embedding model (downloads on first run, ~90MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all chunks...
  Embedded 7731/7731 chunks...
Embedding matrix shape: (7731, 384)


### Why `normalize_embeddings=True`?

When vectors are L2-normalized (length = 1), **cosine similarity equals dot product**.
This is important because FAISS's fastest index (`IndexFlatIP`) uses inner product,
which equals cosine similarity only for normalized vectors. Normalizing at embed time
means faster, simpler retrieval later.

### Alternative Embedding Models

| Model | Dims | Size | Notes |
|---|---|---|---|
| `all-MiniLM-L6-v2` ← *we use* | 384 | ~90MB | Fast, good general quality |
| `all-mpnet-base-v2` | 768 | ~420MB | Slower, better quality |
| `BAAI/bge-large-en-v1.5` | 1024 | ~1.3GB | State-of-the-art open source |
| OpenAI `text-embedding-3-small` | 1536 | API call | Excellent quality, costs money |
| OpenAI `text-embedding-3-large` | 3072 | API call | Best quality, more expensive |

---

## Part 5 — Vector Storage with FAISS

### What FAISS Does

FAISS (Facebook AI Similarity Search) stores your embeddings in an index and lets you
ask: *"which of these N vectors is most similar to this query vector?"* — very fast,
even with millions of vectors.

In [6]:
import faiss
import pickle

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Build a FAISS flat inner-product index.

    IndexFlatIP: brute-force exact search using inner product (= cosine sim
    when vectors are normalized). No approximation — finds the true nearest
    neighbors. Use approximate indexes (IVF, HNSW) for 1M+ vectors.
    """
    dim = embeddings.shape[1]       # embedding dimension, e.g. 384
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product
    index.add(embeddings)           # add all chunk vectors
    return index


def save_index(index, chunks, path_prefix="rag_index"):
    """Save FAISS index + chunk metadata to disk."""
    faiss.write_index(index, f"{path_prefix}.faiss")
    with open(f"{path_prefix}_chunks.pkl", "wb") as f:
        pickle.dump(chunks, f)
    print(f"✓ Saved index ({index.ntotal} vectors) to {path_prefix}.faiss")


def load_index(path_prefix="rag_index"):
    """Load FAISS index + chunk metadata from disk."""
    index = faiss.read_index(f"{path_prefix}.faiss")
    with open(f"{path_prefix}_chunks.pkl", "rb") as f:
        chunks = pickle.load(f)
    print(f"✓ Loaded index ({index.ntotal} vectors)")
    return index, chunks


# Build and save
index = build_faiss_index(embeddings)
save_index(index, chunks)

✓ Saved index (7731 vectors) to rag_index.faiss


### FAISS Index Types Explained

| Index | How it works | When to use |
|---|---|---|
| `IndexFlatIP` ← *we use* | Brute-force exact search | < 500K vectors, max accuracy |
| `IndexIVFFlat` | Cluster vectors, search only nearby clusters | 500K–5M vectors |
| `IndexHNSWFlat` | Graph-based approximate search | Fast queries, high recall |
| `IndexIVFPQ` | Compressed approximate search | Millions of vectors, memory-constrained |

**Alternative vector databases** (when you need persistence + scale):
- **Pinecone** — fully managed, great free tier, best for production
- **Weaviate** — open source, supports hybrid search (vector + keyword)
- **Chroma** — simplest local option, good for prototyping
- **Qdrant** — high performance, great filtering support
- **pgvector** — if you're already on PostgreSQL

---

## Part 6 — Retrieval

### The Retrieval Function

At query time: embed the question → search FAISS → return top-k chunks.

In [7]:
def retrieve(
    query: str,
    index: faiss.Index,
    chunks: list[dict],
    model: SentenceTransformer,
    top_k: int = 5,
) -> list[dict]:
    """
    Embed a query and retrieve the top-k most relevant chunks.

    Returns list of chunk dicts with an added 'score' field.
    Score is cosine similarity (0 to 1, higher = more relevant).
    """
    # Embed the query (same model, same normalization as index)
    query_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Search FAISS: returns (scores, indices) arrays of shape (1, top_k)
    scores, indices = index.search(query_vector, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:     # FAISS returns -1 for empty slots
            continue
        chunk = chunks[idx].copy()
        chunk["score"] = float(score)
        results.append(chunk)

    return results


# Test retrieval
query = "What obligations do providers of high-risk AI systems have?"
results = retrieve(query, index, chunks, model, top_k=5)

print(f"Query: {query}\n")
for i, r in enumerate(results):
    print(f"Result {i+1} | Score: {r['score']:.3f} | "
          f"{r['source_file']} | Page {r['page_number']}")
    print(r["text"][:300])
    print("-" * 60)

Query: What obligations do providers of high-risk AI systems have?

Result 1 | Score: 0.853 | eu_ai_act.pdf | Page 82
(85) General-purpose AI systems may be used as high-risk AI systems by themselves or be 
components of other high-risk AI systems. Therefore, due to their particular nature and 
in order to ensure a fair sharing of responsibilities along the AI value chain, the 
providers of such systems should, irr
------------------------------------------------------------
Result 2 | Score: 0.810 | eu_ai_act.pdf | Page 63
(64) To mitigate the risks from high-risk AI systems placed on the market or put into service 
and to ensure a high level of trustworthiness, certain mandatory requirements should 
apply to high-risk AI systems, taking into account the intended purpose and the context of 
use of the AI system and ac
------------------------------------------------------------
Result 3 | Score: 0.786 | eu_ai_act.pdf | Page 157
uivalent in substance to the notion of 
substantial modif

## What the Score Means

The score is **cosine similarity** (0 to 1):
- `> 0.85` — very strong match, almost certainly relevant
- `0.70–0.85` — good match, likely relevant
- `0.50–0.70` — moderate match, might be relevant
- `< 0.50` — weak match, probably noise

You can add a score threshold to filter out weak results:

```python
results = [r for r in results if r["score"] > 0.60]
```

### Optional: Re-ranking for Higher Precision

Vector similarity is fast but imprecise. A **cross-encoder re-ranker** reads each
(query, chunk) pair together and gives a more accurate relevance score — at the cost
of speed.


In [8]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, results: list[dict], top_k: int = 3) -> list[dict]:
    """Re-rank retrieved chunks using a cross-encoder for higher precision."""
    pairs = [(query, r["text"]) for r in results]
    scores = reranker.predict(pairs)
    for r, s in zip(results, scores):
        r["rerank_score"] = float(s)
    return sorted(results, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

# Usage: retrieve more candidates, then re-rank to top-3
results = retrieve(query, index, chunks, model, top_k=10)
results = rerank(query, results, top_k=3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

**Why retrieve 10 then re-rank to 3?** The embedding model retrieves broadly (recall).
The cross-encoder then precisely selects the best ones (precision). This two-stage
approach is used by production systems like Google's search.

---

## Part 7 — Generation (RAG Answer)

### Building the Prompt

Now we inject the retrieved chunks into an LLM prompt. The structure is:

1. **System instruction** — tell the model to answer only from context
2. **Context block** — the retrieved chunks, clearly labeled
3. **User question** — the actual query

In [9]:
def build_prompt(query: str, results: list[dict]) -> str:
    """
    Construct the RAG prompt by injecting retrieved chunks as context.
    """
    context_parts = []
    for i, r in enumerate(results):
        citation = f"[Source: {r['source_file']}, Page {r['page_number']}]"
        context_parts.append(f"--- Context {i+1} {citation}\n{r['text']}")

    context_block = "\n\n".join(context_parts)

    prompt = f"""You are a legal and regulatory assistant. Answer the user's question
using ONLY the context provided below. If the answer is not found in the context,
say "I cannot find this in the provided documents." Do not use any prior knowledge.
Always cite which source and page number your answer comes from.

{context_block}

Question: {query}

Answer:"""
    return prompt

### Calling the LLM

**Option A: OpenAI API** (best quality, costs money)

In [29]:
from openai import OpenAI

client = OpenAI(api_key="your-api-key-here")

def generate_answer_openai(prompt: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",   # cheap and very capable
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,         # 0 = deterministic, no creativity — good for factual Q&A
        max_tokens=800,
    )
    return response.choices[0].message.content

**Option B: Anthropic Claude API** (excellent at document comprehension)

In [31]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 10.5 MB/s eta 0:00:00


In [32]:
import anthropic

client = anthropic.Anthropic(api_key="your-api-key-here")

def generate_answer_claude(prompt: str) -> str:
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",  # fast and cheap
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

**Option C: Free local model via Ollama** (no API key, runs on your machine)

In [33]:
import requests

def generate_answer_ollama(prompt: str, model: str = "llama3") -> str:
    resp = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
    )
    return resp.json()["response"]

**Option D: Google Gemini**(I am Using Gemini for this )

In [10]:
!pip install -q google-generativeai

In [16]:
import google.generativeai as genai

api_key = os.environ.get("GOOGLE_API_KEY")
genai.configure(api_key=api_key)

# ✅ Use a different name — don't overwrite the embedding model
gemini_model = genai.GenerativeModel("gemini-1.5-pro")

def generate_answer_gemini(prompt: str) -> str:
    response = gemini_model.generate_content(   # ← gemini_model, not model
        prompt,
        generation_config={
            "temperature": 0,
            "max_output_tokens": 800,
        }
    )
    return response.text

---

## Part 8 — Full Pipeline (End-to-End)

Putting all the pieces together into a single callable function:

In [47]:
# ── 1. Clearly named variables, never reuse 'model' ──────────────────────────
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

import anthropic
from openai import OpenAI
import google.generativeai as genai

# ── Named clients, no collisions ever ────────────────────────────────────────
openai_client  = OpenAI(api_key="your-openai-key")
claude_client  = anthropic.Anthropic(api_key="your-anthropic-key")
genai.configure(api_key="Gemini_API_KEY")
gemini_model   = genai.GenerativeModel("gemini-2.5-flash")

# ── Generation functions, each uses its own client ────────────────────────────
def generate_answer_openai(prompt: str) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=800,
    )
    return response.choices[0].message.content

def generate_answer_claude(prompt: str) -> str:
    message = claude_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text

def generate_answer_gemini(prompt: str) -> str:
    response = gemini_model.generate_content(
        prompt,
        generation_config={"temperature": 0, "max_output_tokens": 800}
    )
    return response.text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


None


In [36]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [48]:
def rag_query(
    query: str,
    index: faiss.Index,
    chunks: list[dict],
    model: SentenceTransformer,
    top_k: int = 5,
    use_reranker: bool = False,
    llm: str = "gemini",     # "openai" | "claude" | "ollama" | "gemini"
) -> dict:
    """
    Full RAG pipeline: query → retrieve → rerank (optional) → generate.

    Returns dict with: answer, sources, retrieved_chunks
    """
    # Step 1: Retrieve
    results = retrieve(query, index, chunks, model, top_k=top_k)

    # Step 2 (optional): Re-rank
    if use_reranker:
        results = rerank(query, results, top_k=3)
    else:
        results = results[:3]   # just take top 3

    # Step 3: Build prompt
    prompt = build_prompt(query, results)

    # Step 4: Generate
    if llm == "openai":
        answer = generate_answer_openai(prompt)
    elif llm == "claude":
        answer = generate_answer_claude(prompt)
    elif llm == "ollama":
        answer = generate_answer_ollama(prompt)
    elif llm == "gemini":
        answer = generate_answer_gemini(prompt)
    else:
        raise ValueError(f"Unknown LLM: {llm}")

    # Step 5: Return structured result
    sources = [
        {"file": r["source_file"], "page": r["page_number"], "score": r["score"]}
        for r in results
    ]
    return {"answer": answer, "sources": sources, "chunks_used": results}


# Run it!
result = rag_query(
    query="What are the risk management obligations for high-risk AI systems?",
    index=index,
    chunks=chunks,
    model=model,
    top_k=5,
    use_reranker=False,
    llm="gemini",
)

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"  • {s['file']} | Page {s['page']} | Score {s['score']:.3f}")


ANSWER:
The risk-management system for high-risk AI systems should be a continuous, iterative process planned and run throughout the entire lifecycle of the system (eu_ai_act.pdf, Page 65). This process aims to identify and mitigate relevant risks of AI systems on health, safety, and fundamental rights (eu_ai_act.pdf, Page 65). It must be regularly reviewed and updated to ensure its continuing effectiveness, and significant decisions and actions must be justified and documented (eu_ai_act.pdf, Page 65). Providers must identify risks or adverse impacts and implement mitigation measures for known and reasonably foreseeable risks to health, safety, and fundamental rights, considering the intended purpose and reasonably foreseeable misuse, including risks from interaction with the environment (eu_ai_act.pdf, Page 65).

In identifying the most appropriate risk management measures, the following must be ensured:
*   Elimination or reduction of identified and evaluated risks as far as technic


---

## Part 9 — Evaluation

### Why You Must Evaluate

A RAG system can fail in two independent ways:

1. **Retrieval failure** — the right chunk was never fetched (wrong chunks returned)
2. **Generation failure** — the right chunk was fetched but the LLM ignored or
   misread it

You need to measure both separately, or you cannot debug the system.

In [49]:
# Build a small golden dataset by hand: 10-20 Q&A pairs from your documents
golden_dataset = [
    {
        "query": "What are the prohibited AI practices under the EU AI Act?",
        "expected_keywords": ["biometric", "subliminal", "manipulation", "Article 5"],
        "expected_source": "eu_ai_act.pdf",
    },
    {
        "query": "What does NIST mean by the GOVERN function in the AI RMF?",
        "expected_keywords": ["policies", "organizational", "accountability", "culture"],
        "expected_source": "nist_ai_rmf.pdf",
    },
]

def evaluate(dataset, index, chunks, model):
    retrieval_hits = 0
    for item in dataset:
        results = retrieve(item["query"], index, chunks, model, top_k=5)

        # Retrieval hit: did the correct source appear in top-5?
        sources = [r["source_file"] for r in results]
        if item["expected_source"] in sources:
            retrieval_hits += 1

        # Keyword hit: do retrieved chunks contain expected terms?
        combined_text = " ".join(r["text"] for r in results).lower()
        keyword_hits = sum(
            1 for kw in item["expected_keywords"]
            if kw.lower() in combined_text
        )
        print(f"Query: {item['query'][:60]}...")
        print(f"  Source hit: {'✓' if item['expected_source'] in sources else '✗'}")
        print(f"  Keyword hits: {keyword_hits}/{len(item['expected_keywords'])}")

    recall = retrieval_hits / len(dataset)
    print(f"\nRetrieval recall@5: {recall:.0%}")

evaluate(golden_dataset, index, chunks, model)

Query: What are the prohibited AI practices under the EU AI Act?...
  Source hit: ✓
  Keyword hits: 2/4
Query: What does NIST mean by the GOVERN function in the AI RMF?...
  Source hit: ✓
  Keyword hits: 3/4

Retrieval recall@5: 100%


### Production Evaluation Tools

When you go beyond a golden test set, consider:
- **RAGAS** — automated RAG evaluation framework (faithfulness, relevancy, recall)
- **TruLens** — traces LLM calls and grades them
- **DeepEval** — unit-test style framework for LLM outputs

---

## Architecture Decisions Summary

```
                    ┌─────────────────────────────────────────┐
                    │              RAG PIPELINE               │
                    └─────────────────────────────────────────┘

  OFFLINE (build once, save to disk)
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │  PDFs    │───►│  Parse   │───►│  Chunk   │───►│  Embed   │
  │(source   │    │ (pypdf)  │    │(sliding  │    │(MiniLM)  │
  │documents)│    │          │    │ window)  │    │          │
  └──────────┘    └──────────┘    └──────────┘    └──────┬───┘
                                                         │
                                                         ▼
                                                  ┌──────────────┐
                                                  │ FAISS Index  │
                                                  │ (saved disk) │
                                                  └──────────────┘

  ONLINE (run per query)
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │  User    │───►│  Embed   │───►│  FAISS   │───►│ Re-rank  │───►│   LLM    │
  │  Query   │    │  Query   │    │  Search  │    │(optional)│    │ Generate │
  └──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
                                       │                                │
                                  top-k chunks                    grounded
                                  + scores                         answer
```

---

## Common Failure Modes & Fixes

| Symptom | Likely Cause | Fix |
|---|---|---|
| Retrieves wrong chunks | chunk_size too large, semantic dilution | Reduce chunk_size to 500–800 |
| Misses relevant chunks | chunk_size too small, context cut | Increase to 1200, add more overlap |
| LLM ignores context | Prompt not strict enough | Strengthen system instruction |
| LLM makes up sources | No grounding in prompt | Add: "cite exact source file and page" |
| Slow embedding | Large batch on CPU | Reduce batch_size, use GPU if available |
| PDF text garbled | Complex PDF format | Switch from pypdf to pymupdf |
| Low recall score | Bad embedding model for domain | Try `BAAI/bge-large-en-v1.5` |

---

## What to Build Next

Once your basic pipeline works, here are natural next steps:

1. **Hybrid search** — combine FAISS (semantic) with BM25 (keyword) using Reciprocal Rank Fusion. Keywords catch exact regulatory article numbers (e.g., "Article 9") that semantic search can miss.

2. **Metadata filtering** — filter by source file before searching. "Only search the NIST documents" becomes a pre-filter on the FAISS index.

3. **Conversation memory** — maintain chat history and include prior turns in the context so users can ask follow-up questions.

4. **Streaming responses** — use `stream=True` in OpenAI/Claude APIs so the answer appears token-by-token in the UI instead of all at once.

5. **Document update pipeline** — detect new/changed PDFs, re-embed only the changed chunks, and upsert into the index without rebuilding from scratch.

---

*All code in this guide uses only: `pypdf`, `sentence-transformers`, `faiss-cpu`, `numpy`, `requests`, and an LLM API of your choice. No frameworks required.*
